# CredGuard AI — Home Credit Default Risk: Full Training Pipeline

**Pipeline overview:**
1. Load & audit all 7 source files
2. Feature engineering per table
3. Assemble master training matrix
4. Preprocessing (missing values, encoding, SMOTE+ENN)
5. Train LightGBM with Optuna tuning
6. Train XGBoost with Optuna tuning
7. Stacked ensemble + calibration
8. SHAP explainability
9. Export final model artifacts

**Expected output:** AUC ~0.79–0.81 on 5-fold stratified CV

**Dataset:** Place all Home Credit CSVs in `./data/` before running.

## 0. Install & imports

In [8]:
# Run once to install dependencies
!pip install lightgbm xgboost optuna shap imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib -q


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import joblib
import json
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE

import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED       = 42
N_FOLDS    = 5
DATA_DIR   = '../datasets/home-credit-default-risk'
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(SEED)
print('All imports OK')

All imports OK


## 1. Load source files

In [10]:
def load_csv(name):
    path = os.path.join(DATA_DIR, name)
    df   = pd.read_csv(path)
    print(f'{name:45s}  {df.shape[0]:>8,} rows  {df.shape[1]:>4} cols')
    return df

app_train   = load_csv('application_train.csv')
bureau      = load_csv('bureau.csv')
bb          = load_csv('bureau_balance.csv')
prev        = load_csv('previous_application.csv')
pos         = load_csv('POS_CASH_balance.csv')
cc          = load_csv('credit_card_balance.csv')
inst        = load_csv('installments_payments.csv')

print(f'\nTarget distribution:\n{app_train["TARGET"].value_counts(normalize=True).round(4)}')

application_train.csv                           307,511 rows   122 cols
bureau.csv                                     1,716,428 rows    17 cols
bureau_balance.csv                             27,299,925 rows     3 cols
previous_application.csv                       1,670,214 rows    37 cols
POS_CASH_balance.csv                           10,001,358 rows     8 cols
credit_card_balance.csv                        3,840,312 rows    23 cols
installments_payments.csv                      13,605,401 rows     8 cols

Target distribution:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64


## 2. Feature engineering — bureau + bureau_balance

In [11]:
# ── bureau_balance → aggregate to SK_ID_BUREAU ───────────────────────────────
STATUS_MAP = {'C': 0, 'X': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bb['STATUS_NUM'] = bb['STATUS'].map(STATUS_MAP).fillna(0)

bb_agg = bb.groupby('SK_ID_BUREAU').agg(
    bb_count        = ('MONTHS_BALANCE', 'count'),
    bb_months_max   = ('MONTHS_BALANCE', 'max'),
    bb_dpd_mean     = ('STATUS_NUM', 'mean'),
    bb_dpd_max      = ('STATUS_NUM', 'max'),
    bb_dpd_sum      = ('STATUS_NUM', 'sum'),
    bb_never_late   = ('STATUS_NUM', lambda x: (x == 0).mean()),
    bb_severe_late  = ('STATUS_NUM', lambda x: (x >= 3).mean()),
).reset_index()

# ── bureau → join bb_agg, aggregate to SK_ID_CURR ────────────────────────────
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

# Derived columns before aggregation
bureau['CREDIT_DEBT_RATIO'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] / (bureau['AMT_CREDIT_SUM'] + 1)
)
bureau['CREDIT_OVERDUE_RATIO'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] / (bureau['AMT_CREDIT_SUM'] + 1)
)
bureau['IS_ACTIVE'] = (bureau['CREDIT_ACTIVE'] == 'Active').astype(int)

num_cols = bureau.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['SK_ID_CURR', 'SK_ID_BUREAU']]

agg_dict = {}
for col in num_cols:
    agg_dict[f'bur_{col}_mean'] = (col, 'mean')
    agg_dict[f'bur_{col}_max']  = (col, 'max')
    agg_dict[f'bur_{col}_sum']  = (col, 'sum')

# Add count separately
extra = bureau.groupby('SK_ID_CURR').agg(
    bur_count         = ('SK_ID_BUREAU', 'count'),
    bur_active_count  = ('IS_ACTIVE', 'sum'),
    bur_active_ratio  = ('IS_ACTIVE', 'mean'),
    bur_closed_count  = ('IS_ACTIVE', lambda x: (x == 0).sum()),
).reset_index()

bureau_agg = bureau.groupby('SK_ID_CURR').agg(**agg_dict).reset_index()
bureau_agg = bureau_agg.merge(extra, on='SK_ID_CURR', how='left')

print(f'bureau_agg shape: {bureau_agg.shape}')
del bb, bb_agg, bureau
gc.collect()

bureau_agg shape: (305811, 71)


0

## 3. Feature engineering — previous_application

In [12]:
prev['APP_CREDIT_RATIO']  = prev['AMT_APPLICATION'] / (prev['AMT_CREDIT'] + 1)
prev['CREDIT_GOODS_RATIO']= prev['AMT_CREDIT'] / (prev['AMT_GOODS_PRICE'] + 1)
prev['DOWN_PAY_RATIO']    = prev['AMT_DOWN_PAYMENT'] / (prev['AMT_CREDIT'] + 1)
prev['ANNUITY_CREDIT_RATIO'] = prev['AMT_ANNUITY'] / (prev['AMT_CREDIT'] + 1)

# Flag refused applications
prev['IS_APPROVED']  = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev['IS_REFUSED']   = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
prev['IS_CONSUMER']  = (prev['NAME_CONTRACT_TYPE'] == 'Consumer loans').astype(int)
prev['IS_CASH']      = (prev['NAME_CONTRACT_TYPE'] == 'Cash loans').astype(int)
prev['IS_REVOLVING'] = (prev['NAME_CONTRACT_TYPE'] == 'Revolving loans').astype(int)

num_cols = prev.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['SK_ID_CURR', 'SK_ID_PREV']]

agg_dict = {}
for col in num_cols:
    agg_dict[f'prev_{col}_mean'] = (col, 'mean')
    agg_dict[f'prev_{col}_max']  = (col, 'max')
    agg_dict[f'prev_{col}_min']  = (col, 'min')

prev_agg = prev.groupby('SK_ID_CURR').agg(**agg_dict).reset_index()

extra = prev.groupby('SK_ID_CURR').agg(
    prev_count          = ('SK_ID_PREV', 'count'),
    prev_approved_count = ('IS_APPROVED', 'sum'),
    prev_refused_count  = ('IS_REFUSED', 'sum'),
    prev_approval_rate  = ('IS_APPROVED', 'mean'),
    prev_consumer_count = ('IS_CONSUMER', 'sum'),
    prev_cash_count     = ('IS_CASH', 'sum'),
    prev_revolving_count= ('IS_REVOLVING', 'sum'),
).reset_index()

prev_agg = prev_agg.merge(extra, on='SK_ID_CURR', how='left')

print(f'prev_agg shape: {prev_agg.shape}')
del prev
gc.collect()

prev_agg shape: (338857, 92)


0

## 4. Feature engineering — installments_payments (most behavioral)

In [13]:
# Core behavioral signals — payment timing & completeness
inst['PAYMENT_DIFF']  = inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']
# Positive = paid early, Negative = paid late

inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1e-9)
# > 1.0 = overpaid, < 1.0 = underpaid

inst['IS_LATE']         = (inst['PAYMENT_DIFF'] < 0).astype(int)
inst['IS_EARLY']        = (inst['PAYMENT_DIFF'] > 0).astype(int)
inst['DAYS_LATE']       = (-inst['PAYMENT_DIFF']).clip(lower=0)
inst['PAYMENT_SHORT']   = (inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']).clip(lower=0)
# How much was underpaid

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count              = ('SK_ID_PREV', 'count'),
    inst_late_rate          = ('IS_LATE', 'mean'),
    inst_early_rate         = ('IS_EARLY', 'mean'),
    inst_days_late_mean     = ('DAYS_LATE', 'mean'),
    inst_days_late_max      = ('DAYS_LATE', 'max'),
    inst_days_late_sum      = ('DAYS_LATE', 'sum'),
    inst_pay_ratio_mean     = ('PAYMENT_RATIO', 'mean'),
    inst_pay_ratio_min      = ('PAYMENT_RATIO', 'min'),
    inst_pay_ratio_std      = ('PAYMENT_RATIO', 'std'),
    inst_diff_mean          = ('PAYMENT_DIFF', 'mean'),
    inst_diff_std           = ('PAYMENT_DIFF', 'std'),
    # Consistency: low std = predictable payer
    inst_short_mean         = ('PAYMENT_SHORT', 'mean'),
    inst_short_max          = ('PAYMENT_SHORT', 'max'),
    inst_short_sum          = ('PAYMENT_SHORT', 'sum'),
    inst_amt_payment_mean   = ('AMT_PAYMENT', 'mean'),
    inst_amt_payment_sum    = ('AMT_PAYMENT', 'sum'),
    inst_amt_instalment_mean= ('AMT_INSTALMENT', 'mean'),
    inst_num_versions       = ('NUM_INSTALMENT_VERSION', 'nunique'),
).reset_index()

# Recent behavior: last 12 months only
inst_recent = inst[inst['DAYS_INSTALMENT'] >= -365].groupby('SK_ID_CURR').agg(
    inst_recent_late_rate   = ('IS_LATE', 'mean'),
    inst_recent_days_late   = ('DAYS_LATE', 'mean'),
    inst_recent_pay_ratio   = ('PAYMENT_RATIO', 'mean'),
).reset_index()

inst_agg = inst_agg.merge(inst_recent, on='SK_ID_CURR', how='left')

print(f'inst_agg shape: {inst_agg.shape}')
del inst, inst_recent
gc.collect()

inst_agg shape: (339587, 22)


0

## 5. Feature engineering — POS_CASH_balance

In [14]:
pos['IS_COMPLETED'] = (pos['NAME_CONTRACT_STATUS'] == 'Completed').astype(int)
pos['IS_ACTIVE']    = (pos['NAME_CONTRACT_STATUS'] == 'Active').astype(int)
pos['HAS_DPD']      = (pos['SK_DPD'] > 0).astype(int)

pos_agg = pos.groupby('SK_ID_CURR').agg(
    pos_count               = ('SK_ID_PREV', 'count'),
    pos_months_count        = ('MONTHS_BALANCE', 'count'),
    pos_months_max          = ('MONTHS_BALANCE', 'max'),
    pos_dpd_mean            = ('SK_DPD', 'mean'),
    pos_dpd_max             = ('SK_DPD', 'max'),
    pos_dpd_def_mean        = ('SK_DPD_DEF', 'mean'),
    pos_dpd_def_max         = ('SK_DPD_DEF', 'max'),
    pos_has_dpd_rate        = ('HAS_DPD', 'mean'),
    pos_completed_count     = ('IS_COMPLETED', 'sum'),
    pos_active_count        = ('IS_ACTIVE', 'sum'),
    pos_completed_rate      = ('IS_COMPLETED', 'mean'),
    pos_instalment_future   = ('CNT_INSTALMENT_FUTURE', 'mean'),
).reset_index()

print(f'pos_agg shape: {pos_agg.shape}')
del pos
gc.collect()

pos_agg shape: (337252, 13)


0

## 6. Feature engineering — credit_card_balance

In [15]:
cc['UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
cc['UTILIZATION'] = cc['UTILIZATION'].clip(0, 1)

cc['MIN_PAY_RATIO'] = cc['AMT_PAYMENT_CURRENT'] / (
    cc['AMT_INST_MIN_REGULARITY'] + 1
)
cc['OVERPAY']     = (cc['MIN_PAY_RATIO'] > 1).astype(int)
cc['UNDERPAY']    = (cc['MIN_PAY_RATIO'] < 1).astype(int)
cc['DRAWINGS_RATIO'] = cc['AMT_DRAWINGS_CURRENT'] / (
    cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1
)

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_count                = ('SK_ID_PREV', 'count'),
    cc_months_count         = ('MONTHS_BALANCE', 'count'),
    cc_util_mean            = ('UTILIZATION', 'mean'),
    cc_util_max             = ('UTILIZATION', 'max'),
    cc_util_std             = ('UTILIZATION', 'std'),
    cc_dpd_mean             = ('SK_DPD', 'mean'),
    cc_dpd_max              = ('SK_DPD', 'max'),
    cc_dpd_def_mean         = ('SK_DPD_DEF', 'mean'),
    cc_min_pay_ratio_mean   = ('MIN_PAY_RATIO', 'mean'),
    cc_min_pay_ratio_min    = ('MIN_PAY_RATIO', 'min'),
    cc_overpay_rate         = ('OVERPAY', 'mean'),
    cc_underpay_rate        = ('UNDERPAY', 'mean'),
    cc_drawings_mean        = ('AMT_DRAWINGS_CURRENT', 'mean'),
    cc_drawings_atm_mean    = ('AMT_DRAWINGS_ATM_CURRENT', 'mean'),
    cc_drawings_ratio_mean  = ('DRAWINGS_RATIO', 'mean'),
    cc_balance_mean         = ('AMT_BALANCE', 'mean'),
    cc_balance_max          = ('AMT_BALANCE', 'max'),
    cc_limit_mean           = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
).reset_index()

# Utilization trend: last vs mean
cc_sorted = cc.sort_values('MONTHS_BALANCE')
cc_last   = cc_sorted.groupby('SK_ID_CURR')['UTILIZATION'].last().reset_index()
cc_last.columns = ['SK_ID_CURR', 'cc_util_last']
cc_agg    = cc_agg.merge(cc_last, on='SK_ID_CURR', how='left')
cc_agg['cc_util_trend'] = cc_agg['cc_util_last'] - cc_agg['cc_util_mean']

print(f'cc_agg shape: {cc_agg.shape}')
del cc, cc_sorted, cc_last
gc.collect()

cc_agg shape: (103558, 21)


0

## 7. Feature engineering — application_train (base table)

In [16]:
df = app_train.copy()

# ── Fix known anomalies ───────────────────────────────────────────────────────
# DAYS_EMPLOYED = 365243 means unemployed/retired — flag and null it
df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED']      = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# ── Core ratio features ───────────────────────────────────────────────────────
df['CREDIT_INCOME_RATIO']    = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] + 1)
df['ANNUITY_INCOME_RATIO']   = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
df['CREDIT_TERM_MONTHS']     = df['AMT_CREDIT']   / (df['AMT_ANNUITY'] + 1)
df['GOODS_CREDIT_RATIO']     = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)
df['INCOME_PER_PERSON']      = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
df['CHILDREN_RATIO']         = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)

# ── Age / tenure features ─────────────────────────────────────────────────────
df['AGE_YEARS']              = -df['DAYS_BIRTH'] / 365.25
df['EMPLOYED_YEARS']         = -df['DAYS_EMPLOYED'] / 365.25
df['DAYS_EMPLOYED_RATIO']    = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['ID_PUBLISH_AGE_RATIO']   = df['DAYS_ID_PUBLISH'] / df['DAYS_BIRTH']
df['REG_AGE_RATIO']          = df['DAYS_REGISTRATION'] / df['DAYS_BIRTH']

# ── EXT_SOURCE — most predictive features in the dataset ─────────────────────
ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
df['EXT_SOURCE_MEAN']  = df[ext].mean(axis=1)
df['EXT_SOURCE_PROD']  = df[ext].prod(axis=1)
df['EXT_SOURCE_STD']   = df[ext].std(axis=1)
df['EXT_SOURCE_MIN']   = df[ext].min(axis=1)
df['EXT_SOURCE_MAX']   = df[ext].max(axis=1)
df['EXT12_RATIO']      = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_2'] + 1e-9)
df['EXT23_RATIO']      = df['EXT_SOURCE_2'] / (df['EXT_SOURCE_3'] + 1e-9)
df['EXT_ANNUITY_X2']   = df['EXT_SOURCE_2'] * df['ANNUITY_INCOME_RATIO']
df['EXT_CREDIT_X2']    = df['EXT_SOURCE_2'] * df['CREDIT_INCOME_RATIO']

# ── Document flags (aggregate instead of using all 20 individually) ───────────
doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
df['DOCS_PROVIDED'] = df[doc_cols].sum(axis=1)
df.drop(columns=doc_cols, inplace=True)

# ── Contact flags ─────────────────────────────────────────────────────────────
contact_cols = ['FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE',
                'FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL']
df['CONTACT_COUNT'] = df[contact_cols].sum(axis=1)

# ── Address mismatch flags ────────────────────────────────────────────────────
df['ADDRESS_MISMATCH'] = (
    df['REG_CITY_NOT_LIVE_CITY'].fillna(0) +
    df['REG_CITY_NOT_WORK_CITY'].fillna(0) +
    df['LIVE_CITY_NOT_WORK_CITY'].fillna(0)
)

# ── Drop low-value columns ────────────────────────────────────────────────────
drop_cols = [
    'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY',
    'FLAG_MOBIL','FLAG_EMP_PHONE','FLAG_WORK_PHONE',
    'FLAG_CONT_MOBILE','FLAG_PHONE','FLAG_EMAIL',
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

print(f'Base table shape after engineering: {df.shape}')

Base table shape after engineering: (307511, 115)


## 8. Assemble master training matrix

In [17]:
df = (
    df
    .merge(bureau_agg, on='SK_ID_CURR', how='left')
    .merge(prev_agg,   on='SK_ID_CURR', how='left')
    .merge(inst_agg,   on='SK_ID_CURR', how='left')
    .merge(pos_agg,    on='SK_ID_CURR', how='left')
    .merge(cc_agg,     on='SK_ID_CURR', how='left')
)

print(f'Master matrix shape: {df.shape}')
print(f'Missing rate summary:')
miss = (df.isnull().mean() * 100).sort_values(ascending=False)
print(miss[miss > 50].head(20))  # columns with >50% missing

# Drop columns with >80% missing — too sparse to be reliable
high_miss = miss[miss > 80].index.tolist()
df.drop(columns=high_miss, inplace=True)
print(f'\nDropped {len(high_miss)} columns with >80% missing')
print(f'Final shape: {df.shape}')

del bureau_agg, prev_agg, inst_agg, pos_agg, cc_agg, app_train
gc.collect()

Master matrix shape: (307511, 329)
Missing rate summary:
prev_RATE_INTEREST_PRIMARY_min        98.501192
prev_RATE_INTEREST_PRIVILEGED_min     98.501192
prev_RATE_INTEREST_PRIVILEGED_max     98.501192
prev_RATE_INTEREST_PRIVILEGED_mean    98.501192
prev_RATE_INTEREST_PRIMARY_mean       98.501192
prev_RATE_INTEREST_PRIMARY_max        98.501192
cc_min_pay_ratio_min                  80.143800
cc_min_pay_ratio_mean                 80.143800
cc_drawings_atm_mean                  80.117784
bur_AMT_ANNUITY_mean                  73.981744
bur_AMT_ANNUITY_max                   73.981744
cc_util_std                           71.944743
cc_util_trend                         71.739222
cc_util_last                          71.739222
cc_limit_mean                         71.739222
cc_underpay_rate                      71.739222
cc_drawings_mean                      71.739222
cc_overpay_rate                       71.739222
cc_dpd_max                            71.739222
cc_balance_max                 

0

## 9. Preprocessing — encoding, imputation

In [18]:
# ── Separate target ───────────────────────────────────────────────────────────
y  = df['TARGET'].astype(int)
df = df.drop(columns=['TARGET', 'SK_ID_CURR'])

# ── Label encode binary categoricals ─────────────────────────────────────────
cat_cols  = df.select_dtypes(include=['object']).columns.tolist()
label_enc = {}

for col in cat_cols:
    le = LabelEncoder()
    # Fill NaN with 'Unknown' before encoding
    df[col] = df[col].fillna('Unknown')
    df[col] = le.fit_transform(df[col])
    label_enc[col] = le

print(f'Label-encoded {len(cat_cols)} categorical columns')

# ── Replace inf values ────────────────────────────────────────────────────────
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# ── Save feature names before any numpy conversion ───────────────────────────
feature_names = df.columns.tolist()
print(f'Total features: {len(feature_names)}')
print(f'Remaining missing: {df.isnull().sum().sum():,} values across {df.isnull().any().sum()} columns')
print('\nNote: LightGBM & XGBoost handle NaN natively — no imputation needed for tree models')
print('NaN = "applicant has no history in this table" — distinct from 0 — keep as NaN')

Label-encoded 15 categorical columns
Total features: 318
Remaining missing: 21,089,303 values across 282 columns

Note: LightGBM & XGBoost handle NaN natively — no imputation needed for tree models
NaN = "applicant has no history in this table" — distinct from 0 — keep as NaN


In [19]:
# ── Save the master dataset ───────────────────────────────────────────────────
X = df.values.astype(np.float32)

np.save(os.path.join(OUTPUT_DIR, 'X.npy'), X)
np.save(os.path.join(OUTPUT_DIR, 'y.npy'), y.values)
with open(os.path.join(OUTPUT_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_names, f)

print(f'Saved X.npy  shape: {X.shape}')
print(f'Saved y.npy  shape: {y.shape}')
print(f'Class balance: {y.mean():.3f} default rate')

Saved X.npy  shape: (307511, 318)
Saved y.npy  shape: (307511,)
Class balance: 0.081 default rate


## 10. Cross-validation utilities

In [20]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def run_cv_lgbm(params, X, y, n_splits=5, seed=42):
    """5-fold stratified CV for LightGBM, returns mean OOF AUC."""
    oof_preds = np.zeros(len(y))
    skf_      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in skf_.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        oof_preds[val_idx] = m.predict_proba(X_val)[:, 1]
    return roc_auc_score(y, oof_preds)

def run_cv_xgb(params, X, y, n_splits=5, seed=42):
    """5-fold stratified CV for XGBoost, returns mean OOF AUC."""
    oof_preds = np.zeros(len(y))
    skf_      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in skf_.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        m = xgb.XGBClassifier(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        oof_preds[val_idx] = m.predict_proba(X_val)[:, 1]
    return roc_auc_score(y, oof_preds)

print('CV utilities ready')

CV utilities ready


## 11. Optuna tuning — LightGBM

In [ ]:
# ── Subsample for faster Optuna search ───────────────────────────────────────
# Use 30% of data for hyperparameter search, then retrain on full data
from sklearn.model_selection import train_test_split

X_opt, _, y_opt, _ = train_test_split(
    X, y.values, test_size=0.7, stratify=y.values, random_state=SEED
)
print(f'Optuna search on: {X_opt.shape[0]:,} samples')

n_pos = (y_opt == 1).sum()
n_neg = (y_opt == 0).sum()
SCALE_POS_WEIGHT = n_neg / n_pos
print(f'scale_pos_weight: {SCALE_POS_WEIGHT:.2f}')

Optuna search on: 92,253 samples
scale_pos_weight: 11.39


In [22]:
def lgbm_objective(trial):
    params = {
        # ── Tune these ──────────────────────────────────────────────────────
        'num_leaves':       trial.suggest_int('num_leaves', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.15, 0.6),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
        'min_child_samples':trial.suggest_int('min_child_samples', 20, 300),
        'lambda_l2':        trial.suggest_float('lambda_l2', 1e-2, 20.0, log=True),
        'lambda_l1':        trial.suggest_float('lambda_l1', 1e-3, 5.0, log=True),
        'max_depth':        trial.suggest_int('max_depth', 4, 10),
        # ── Fixed ─────────────────────────────────────────────────────────
        'objective':       'binary',
        'metric':          'auc',
        'boosting_type':   'gbdt',
        'bagging_freq':     1,
        'n_estimators':    2000,
        'learning_rate':   0.05,
        'is_unbalance':    True,
        'random_state':    SEED,
        'verbose':         -1,
        'n_jobs':          -1,
    }
    return run_cv_lgbm(params, X_opt, y_opt, n_splits=3, seed=SEED)

print('Running LightGBM Optuna search (100 trials × 3-fold, ~10–20 min)...')
lgbm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=100, show_progress_bar=True)

print(f'\nBest LightGBM AUC: {lgbm_study.best_value:.5f}')
print(f'Best params: {lgbm_study.best_params}')

Running LightGBM Optuna search (100 trials × 3-fold, ~10–20 min)...


Best trial: 98. Best value: 0.778289: 100%|██████████| 100/100 [44:46<00:00, 26.86s/it] 


Best LightGBM AUC: 0.77829
Best params: {'num_leaves': 111, 'feature_fraction': 0.20481706795072116, 'bagging_fraction': 0.7815857169366087, 'min_child_samples': 279, 'lambda_l2': 2.164801738495242, 'lambda_l1': 0.0019244924978188604, 'max_depth': 4}


## 12. Optuna tuning — XGBoost

In [ ]:
def xgb_objective(trial):
    params = {
        # ── Tune these ──────────────────────────────────────────────────────
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'min_child_weight':  trial.suggest_int('min_child_weight', 10, 150),
        'subsample':         trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.15, 0.6),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-2, 20.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'gamma':             trial.suggest_float('gamma', 0.0, 2.0),
        # ── Fixed ─────────────────────────────────────────────────────────
        'objective':         'binary:logistic',
        'eval_metric':       'auc',
        'tree_method':       'hist',
        'n_estimators':       2000,
        'learning_rate':      0.05,
        'scale_pos_weight':   SCALE_POS_WEIGHT,
        'early_stopping_rounds': 100,
        'random_state':       SEED,
        'n_jobs':            -1,
        'verbosity':          0,
    }
    return run_cv_xgb(params, X_opt, y_opt, n_splits=3, seed=SEED)

print('Running XGBoost Optuna search (80 trials × 3-fold, ~15–25 min)...')
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=80, show_progress_bar=True)

print(f'\nBest XGBoost AUC: {xgb_study.best_value:.5f}')
print(f'Best params: {xgb_study.best_params}')

Running XGBoost Optuna search (80 trials × 3-fold, ~15–25 min)...


Best trial: 3. Best value: 0.778698:   6%|▋         | 5/80 [06:35<1:43:38, 82.91s/it] 

## 13. Final training — full 5-fold CV with best params (multi-seed)

In [ ]:
# ── Build final param dicts from Optuna results ───────────────────────────────
LGBM_FINAL_PARAMS = {
    **lgbm_study.best_params,
    'objective':     'binary',
    'metric':        'auc',
    'boosting_type': 'gbdt',
    'bagging_freq':   1,
    'n_estimators':  3000,      # more trees at final training
    'learning_rate': 0.03,      # lower LR for final = more stable
    'is_unbalance':  True,
    'verbose':       -1,
    'n_jobs':        -1,
}

XGB_FINAL_PARAMS = {
    **xgb_study.best_params,
    'objective':         'binary:logistic',
    'eval_metric':       'auc',
    'tree_method':       'hist',
    'n_estimators':       3000,
    'learning_rate':      0.03,
    'scale_pos_weight':   SCALE_POS_WEIGHT,
    'early_stopping_rounds': 150,
    'random_state':       SEED,
    'n_jobs':            -1,
    'verbosity':          0,
}

SEEDS = [42, 43, 44, 45, 46]  # multi-seed averaging

print('Final LGBM params:')
print(json.dumps({k: v for k, v in LGBM_FINAL_PARAMS.items() if k in lgbm_study.best_params}, indent=2))
print('\nFinal XGB params:')
print(json.dumps({k: v for k, v in XGB_FINAL_PARAMS.items() if k in xgb_study.best_params}, indent=2))

In [ ]:
y_np    = y.values if hasattr(y, 'values') else y
skf_fin = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage
lgbm_oof_preds  = np.zeros(len(y_np))
xgb_oof_preds   = np.zeros(len(y_np))
lgbm_models     = []
xgb_models      = []
fold_aucs_lgbm  = []
fold_aucs_xgb   = []

print(f'Training {N_FOLDS}-fold × {len(SEEDS)} seeds...\n')

for fold, (tr_idx, val_idx) in enumerate(skf_fin.split(X, y_np)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y_np[tr_idx], y_np[val_idx]

    # ── LightGBM: average over seeds ─────────────────────────────────────────
    lgbm_fold_preds = np.zeros(len(val_idx))
    fold_lgbm_models = []
    for seed in SEEDS:
        params = {**LGBM_FINAL_PARAMS, 'random_state': seed}
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(150, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        lgbm_fold_preds += m.predict_proba(X_val)[:, 1] / len(SEEDS)
        fold_lgbm_models.append(m)

    lgbm_oof_preds[val_idx] = lgbm_fold_preds
    lgbm_models.append(fold_lgbm_models)
    fold_auc = roc_auc_score(y_val, lgbm_fold_preds)
    fold_aucs_lgbm.append(fold_auc)

    # ── XGBoost: average over seeds ──────────────────────────────────────────
    xgb_fold_preds = np.zeros(len(val_idx))
    fold_xgb_models = []
    for seed in SEEDS:
        params = {**XGB_FINAL_PARAMS, 'random_state': seed}
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        xgb_fold_preds += m.predict_proba(X_val)[:, 1] / len(SEEDS)
        fold_xgb_models.append(m)

    xgb_oof_preds[val_idx] = xgb_fold_preds
    xgb_models.append(fold_xgb_models)
    fold_auc = roc_auc_score(y_val, xgb_fold_preds)
    fold_aucs_xgb.append(fold_auc)

    print(f'Fold {fold+1} | LGBM AUC: {fold_aucs_lgbm[-1]:.5f}  |  XGB AUC: {fold_aucs_xgb[-1]:.5f}')

lgbm_oof_auc = roc_auc_score(y_np, lgbm_oof_preds)
xgb_oof_auc  = roc_auc_score(y_np, xgb_oof_preds)
print(f'\n=== OOF AUC ===')
print(f'LightGBM: {lgbm_oof_auc:.5f}  (std: {np.std(fold_aucs_lgbm):.5f})')
print(f'XGBoost:  {xgb_oof_auc:.5f}  (std: {np.std(fold_aucs_xgb):.5f})')

## 14. Ensemble — weighted blend + stacked meta-model

In [ ]:
# ── Sweep blend weights to find optimal ratio ─────────────────────────────────
best_w, best_auc = 0.5, 0.0
for w in np.arange(0.3, 0.8, 0.05):
    blended = w * lgbm_oof_preds + (1 - w) * xgb_oof_preds
    auc     = roc_auc_score(y_np, blended)
    if auc > best_auc:
        best_auc, best_w = auc, w

LGBM_WEIGHT = round(best_w, 2)
XGB_WEIGHT  = round(1 - best_w, 2)
ensemble_oof = LGBM_WEIGHT * lgbm_oof_preds + XGB_WEIGHT * xgb_oof_preds
ensemble_auc = roc_auc_score(y_np, ensemble_oof)

print(f'Optimal weights — LGBM: {LGBM_WEIGHT}  XGB: {XGB_WEIGHT}')
print(f'Blend AUC: {ensemble_auc:.5f}  (vs LGBM alone: {lgbm_oof_auc:.5f})')

# ── Stacked meta-model (Logistic Regression on OOF predictions) ───────────────
meta_X  = np.column_stack([lgbm_oof_preds, xgb_oof_preds])
meta_lr = LogisticRegression(C=1.0, max_iter=1000)
meta_lr.fit(meta_X, y_np)
stacked_oof = meta_lr.predict_proba(meta_X)[:, 1]
stacked_auc = roc_auc_score(y_np, stacked_oof)

print(f'\nStacked meta-model AUC: {stacked_auc:.5f}')

# Use whichever is best as final ensemble
FINAL_OOF = stacked_oof if stacked_auc >= ensemble_auc else ensemble_oof
FINAL_AUC = max(stacked_auc, ensemble_auc)
print(f'\nFinal ensemble AUC: {FINAL_AUC:.5f}')

## 15. Probability calibration

In [ ]:
# Calibrate so p_default maps to true empirical default probability
# Important for CredGuard: we convert p_default → readiness score
# Uncalibrated trees are overconfident — calibration fixes this

from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, preds) in zip(axes, [
    ('LightGBM OOF',  lgbm_oof_preds),
    ('Final Ensemble', FINAL_OOF)
]):
    fraction_pos, mean_pred = calibration_curve(y_np, preds, n_bins=15)
    ax.plot(mean_pred, fraction_pos, 's-', label=name)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'Calibration — {name}')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration_curve.png'), dpi=150)
plt.show()

# Isotonic calibration using best LightGBM fold model
# In production: wrap each fold model in CalibratedClassifierCV
print('Calibration plots saved. Apply isotonic regression at serving time.')
print('readiness_score = round((1 - p_default_calibrated) * 100)')

## 16. SHAP analysis — feature importance + explainability

In [ ]:
# Use first fold's first-seed LightGBM model for SHAP
shap_model = lgbm_models[0][0]

# Sample 5000 rows for faster SHAP computation
sample_idx = np.random.choice(len(X), size=min(5000, len(X)), replace=False)
X_shap     = X[sample_idx]

explainer   = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_shap)

# For binary classification, LightGBM returns list [class0, class1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # class 1 = default
else:
    shap_vals = shap_values

# ── Summary plot ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_shap,
    feature_names=feature_names,
    max_display=25,
    show=False
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Top features by mean |SHAP| ───────────────────────────────────────────────
mean_shap = np.abs(shap_vals).mean(axis=0)
top_features = pd.DataFrame({
    'feature':    feature_names,
    'importance': mean_shap
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('\nTop 20 features by mean |SHAP|:')
print(top_features.head(20).to_string(index=False))
top_features.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)

## 17. Per-user SHAP explanation — the CredGuard explanation engine

In [ ]:
def explain_user(user_idx, model, X, feature_names, explainer, top_n=5):
    """
    For a given user, returns:
    - readiness_score (0-100)
    - top factors helping the score
    - top factors hurting the score

    This is the core of CredGuard's explanation UI.
    """
    x     = X[user_idx:user_idx+1]
    p_def = model.predict_proba(x)[0, 1]
    score = round((1 - p_def) * 100)

    sv    = explainer.shap_values(x)
    if isinstance(sv, list):
        sv = sv[1]
    sv = sv[0]

    factors = pd.DataFrame({
        'feature': feature_names,
        'shap':    sv,
        'value':   x[0]
    }).sort_values('shap')

    # Negative SHAP = increases default risk = hurts readiness
    hurting = factors.head(top_n)
    helping = factors.tail(top_n).iloc[::-1]

    return {
        'readiness_score': score,
        'p_default':       round(float(p_def), 4),
        'helping_factors': helping[['feature','shap','value']].to_dict('records'),
        'hurting_factors': hurting[['feature','shap','value']].to_dict('records'),
    }

# Demo: explain first 3 users
for i in range(3):
    result = explain_user(i, shap_model, X_shap, feature_names, explainer)
    print(f"\n── User {i} ─────────────────────────────────")
    print(f"  Readiness score : {result['readiness_score']}/100")
    print(f"  P(default)      : {result['p_default']}")
    print(f"  Top hurting:")
    for f in result['hurting_factors'][:3]:
        print(f"    {f['feature']:40s} SHAP: {f['shap']:+.4f}")
    print(f"  Top helping:")
    for f in result['helping_factors'][:3]:
        print(f"    {f['feature']:40s} SHAP: {f['shap']:+.4f}")

## 18. Simulation engine — what-if score changes

In [ ]:
def simulate_score(user_idx, model, X, feature_names, changes: dict):
    """
    Simulates score change if user modifies specific features.

    changes: dict of {feature_name: new_value}
    e.g. {'ANNUITY_INCOME_RATIO': 0.15, 'CREDIT_INCOME_RATIO': 2.5}

    Returns original score, simulated score, and delta.
    """
    x_orig = X[user_idx:user_idx+1].copy()
    x_sim  = x_orig.copy()

    for feat, new_val in changes.items():
        if feat in feature_names:
            idx          = feature_names.index(feat)
            x_sim[0, idx] = new_val
        else:
            print(f'Warning: feature {feat} not found')

    p_orig = model.predict_proba(x_orig)[0, 1]
    p_sim  = model.predict_proba(x_sim)[0, 1]

    score_orig = round((1 - p_orig) * 100)
    score_sim  = round((1 - p_sim)  * 100)

    return {
        'original_score': score_orig,
        'simulated_score': score_sim,
        'delta': score_sim - score_orig,
        'changes_applied': changes
    }

# Demo simulation: what if user reduces annuity-to-income ratio?
sim = simulate_score(
    user_idx=0,
    model=shap_model,
    X=X_shap,
    feature_names=feature_names,
    changes={
        'ANNUITY_INCOME_RATIO': 0.10,   # was higher, now reduced
        'CREDIT_INCOME_RATIO':  2.0,    # reduced loan ask
    }
)
print(f"\nSimulation result:")
print(f"  Original score  : {sim['original_score']}/100")
print(f"  Simulated score : {sim['simulated_score']}/100")
print(f"  Delta           : {sim['delta']:+d} points")

## 19. Performance summary & ROC curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC curves ────────────────────────────────────────────────────────────────
ax = axes[0]
for name, preds in [
    ('LightGBM OOF',   lgbm_oof_preds),
    ('XGBoost OOF',    xgb_oof_preds),
    ('Ensemble',       FINAL_OOF),
]:
    fpr, tpr, _ = roc_curve(y_np, preds)
    auc_val     = roc_auc_score(y_np, preds)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.4f})')

ax.plot([0, 1], [0, 1], 'k--')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — 5-fold OOF')
ax.legend()

# ── Score distribution by class ───────────────────────────────────────────────
ax = axes[1]
readiness_scores = ((1 - FINAL_OOF) * 100).round()
ax.hist(readiness_scores[y_np == 0], bins=40, alpha=0.6,
        density=True, label='Non-default (repaid)', color='steelblue')
ax.hist(readiness_scores[y_np == 1], bins=40, alpha=0.6,
        density=True, label='Default', color='tomato')
ax.set_xlabel('Readiness Score (0-100)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Outcome')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_performance.png'), dpi=150)
plt.show()

# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '='*50)
print('CREDGUARD MODEL TRAINING SUMMARY')
print('='*50)
print(f'Training samples     : {X.shape[0]:,}')
print(f'Features             : {X.shape[1]:,}')
print(f'Default rate         : {y_np.mean():.2%}')
print(f'LightGBM OOF AUC     : {lgbm_oof_auc:.5f}')
print(f'XGBoost OOF AUC      : {xgb_oof_auc:.5f}')
print(f'Ensemble AUC         : {FINAL_AUC:.5f}')
print(f'Blend weights        : LGBM={LGBM_WEIGHT}  XGB={XGB_WEIGHT}')
print('='*50)

## 20. Save all artifacts

In [ ]:
# Save all fold models
for fold_idx, fold_models in enumerate(lgbm_models):
    for seed_idx, m in enumerate(fold_models):
        path = os.path.join(OUTPUT_DIR, f'lgbm_fold{fold_idx}_seed{seed_idx}.pkl')
        joblib.dump(m, path)

for fold_idx, fold_models in enumerate(xgb_models):
    for seed_idx, m in enumerate(fold_models):
        path = os.path.join(OUTPUT_DIR, f'xgb_fold{fold_idx}_seed{seed_idx}.pkl')
        joblib.dump(m, path)

# Save meta-model
joblib.dump(meta_lr, os.path.join(OUTPUT_DIR, 'meta_model.pkl'))

# Save label encoders
joblib.dump(label_enc, os.path.join(OUTPUT_DIR, 'label_encoders.pkl'))

# Save OOF predictions
oof_df = pd.DataFrame({
    'lgbm_oof':     lgbm_oof_preds,
    'xgb_oof':      xgb_oof_preds,
    'ensemble_oof': FINAL_OOF,
    'target':       y_np,
    'readiness':    ((1 - FINAL_OOF) * 100).round().astype(int)
})
oof_df.to_csv(os.path.join(OUTPUT_DIR, 'oof_predictions.csv'), index=False)

# Save model config
model_config = {
    'lgbm_params':   LGBM_FINAL_PARAMS,
    'xgb_params':    XGB_FINAL_PARAMS,
    'lgbm_weight':   LGBM_WEIGHT,
    'xgb_weight':    XGB_WEIGHT,
    'n_features':    len(feature_names),
    'n_folds':       N_FOLDS,
    'seeds':         SEEDS,
    'lgbm_oof_auc':  float(lgbm_oof_auc),
    'xgb_oof_auc':   float(xgb_oof_auc),
    'ensemble_auc':  float(FINAL_AUC),
}
with open(os.path.join(OUTPUT_DIR, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, indent=2, default=str)

print(f'All artifacts saved to {OUTPUT_DIR}/')
print('\nOutput files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:<45s} {size:>8.1f} KB')